# 07 - Documentos e Textos por Processo STJ

Objetivo: construir a camada textual do corpus, com uma linha por documento e, opcionalmente, uma linha agregada por processo.

Este notebook parte da chave `SeqDocumento` dos metadados de integras e dos arquivos `.txt` dentro do ZIP. O resultado deve preservar a granularidade documento-texto para evitar misturar documentos distintos do mesmo processo.

## 1. Ambiente e parametros

In [ ]:
# Rode esta celula no Colab se necessario.
# !pip install -r requirements.txt

In [1]:
from __future__ import annotations

import json
import re
import sys
import zipfile
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 180)

print('Imports loaded.')

from bs4 import BeautifulSoup

Imports loaded.


In [2]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'notebooks').exists() and (candidate / 'src').exists():
            return candidate
        if candidate.name == 'datajud_probe':
            return candidate
    return start

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PROJECT_ROOT = find_project_root(Path.cwd())

if IN_COLAB:
    MOUNT_POINT = Path('/content/drive')
    if (MOUNT_POINT / 'MyDrive').exists():
        print('Google Drive ja esta montado.')
    else:
        drive.mount(str(MOUNT_POINT), force_remount=True)
    DATA_ROOT = MOUNT_POINT / 'MyDrive/Mestrado/2026/llms/data'
else:
    DATA_ROOT = PROJECT_ROOT / 'data'

API_DATA = PROJECT_ROOT / 'data' / 'api'
RAW_DATA = DATA_ROOT / 'raw'
RAW_STJ = RAW_DATA / 'stj_integras'
METADATA_RAW = RAW_DATA / 'stj_integras_metadata'
PROCESSED_DATA = DATA_ROOT / 'processed'
REPORTS_DATA = DATA_ROOT / 'reports' / 'summaries'
FIGURES_DATA = DATA_ROOT / 'reports' / 'figures'
REFERENCE_DATA = DATA_ROOT / 'reference'

for path in [PROCESSED_DATA, REPORTS_DATA, FIGURES_DATA, REFERENCE_DATA]:
    path.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('API_DATA:', API_DATA, 'exists=', API_DATA.exists())
print('RAW_STJ:', RAW_STJ, 'exists=', RAW_STJ.exists())
print('METADATA_RAW:', METADATA_RAW, 'exists=', METADATA_RAW.exists())
print('PROCESSED_DATA:', PROCESSED_DATA)

Google Drive ja esta montado.
PROJECT_ROOT: /content
DATA_ROOT: /content/drive/MyDrive/Mestrado/2026/llms/data
API_DATA: /content/data/api exists= False
RAW_STJ: /content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras exists= True
METADATA_RAW: /content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras_metadata exists= True
PROCESSED_DATA: /content/drive/MyDrive/Mestrado/2026/llms/data/processed


In [3]:
# Parametros de execucao.
ANOS_ANALISE = [2024, 2025, 2026]
MAX_LOTES = None  # ex.: 2 para testar poucos pares JSON/ZIP; None = todos os lotes encontrados

# Use None para processar todos os documentos encontrados. Para teste, use um numero pequeno, como 500.
MAX_DOCUMENTOS = None
RANDOM_SAMPLE = False
RANDOM_STATE = 42

# Gera uma segunda tabela com textos concatenados por processo. Pode ficar grande.
GERAR_TEXTO_POR_PROCESSO = True

print('ANOS_ANALISE:', ANOS_ANALISE)
print('MAX_LOTES:', MAX_LOTES)
print('MAX_DOCUMENTOS:', MAX_DOCUMENTOS)
print('RANDOM_SAMPLE:', RANDOM_SAMPLE)
print('GERAR_TEXTO_POR_PROCESSO:', GERAR_TEXTO_POR_PROCESSO)

ANOS_ANALISE: [2024, 2025, 2026]
MAX_LOTES: None
MAX_DOCUMENTOS: None
RANDOM_SAMPLE: False
GERAR_TEXTO_POR_PROCESSO: True


## 2. Localizar lotes de metadados e ZIP

Este notebook processa vários lotes. Ele cruza:

- `raw/stj_integras_metadata/`: JSONs `metadadosYYYYMMDD.json`, normalmente baixados pelo notebook 00;
- `raw/stj_integras/`: ZIPs `textosYYYYMMDD.zip` ou `YYYYMMDD.zip`, baixados pelo notebook 00b;
- `processed/stj_processos_ciclo_vida.parquet`: espinha dorsal opcional gerada pelo notebook 06.


In [4]:
def resource_date_from_name(name: str) -> pd.Timestamp | None:
    patterns = [
        r'metadados(\d{6}|\d{8})\.json',
        r'textos(\d{6}|\d{8})\.zip',
        r'(\d{6}|\d{8})\.zip',
    ]
    for pattern in patterns:
        match = re.fullmatch(pattern, name)
        if match:
            raw = match.group(1)
            fmt = '%Y%m%d' if len(raw) == 8 else '%Y%m'
            return pd.to_datetime(raw, format=fmt, errors='coerce')
    return None


def resource_key_from_name(name: str) -> str | None:
    dt = resource_date_from_name(name)
    if dt is None or pd.isna(dt):
        return None
    raw_match = re.search(r'(\d{6}|\d{8})', name)
    return raw_match.group(1) if raw_match else dt.strftime('%Y%m%d')


def wanted_year(path: Path, anos: list[int] | None) -> bool:
    if anos is None:
        return True
    dt = resource_date_from_name(path.name)
    return dt is not None and not pd.isna(dt) and int(dt.year) in anos


def collect_metadata_paths(metadata_root: Path, raw_stj: Path, anos: list[int] | None) -> list[Path]:
    candidates = []
    for root in [metadata_root, raw_stj]:
        if root.exists():
            candidates.extend(root.rglob('metadados*.json'))
    return sorted({p for p in candidates if p.is_file() and wanted_year(p, anos)}, key=lambda p: (resource_key_from_name(p.name) or '', str(p)))


def collect_zip_paths(raw_stj: Path, anos: list[int] | None) -> list[Path]:
    if not raw_stj.exists():
        return []
    candidates = [p for p in raw_stj.rglob('*.zip') if p.is_file()]
    return sorted({p for p in candidates if wanted_year(p, anos)}, key=lambda p: (resource_key_from_name(p.name) or '', str(p)))


def build_lote_table(raw_stj: Path, metadata_root: Path, anos: list[int] | None) -> pd.DataFrame:
    metadata_paths = collect_metadata_paths(metadata_root, raw_stj, anos)
    zip_paths = collect_zip_paths(raw_stj, anos)

    metadata_by_key = {}
    for path in metadata_paths:
        key = resource_key_from_name(path.name)
        if key and key not in metadata_by_key:
            metadata_by_key[key] = path

    zip_by_key = {}
    for path in zip_paths:
        key = resource_key_from_name(path.name)
        if key and key not in zip_by_key:
            zip_by_key[key] = path

    keys = sorted(set(metadata_by_key) | set(zip_by_key))
    rows = []
    for key in keys:
        metadata_path = metadata_by_key.get(key)
        zip_path = zip_by_key.get(key)
        dt = resource_date_from_name(metadata_path.name if metadata_path else zip_path.name)
        rows.append({
            'lote': key,
            'data_lote': dt,
            'ano_lote': int(dt.year) if dt is not None and not pd.isna(dt) else pd.NA,
            'metadata_path': metadata_path,
            'zip_path': zip_path,
            'tem_metadata': metadata_path is not None,
            'tem_zip': zip_path is not None,
        })
    return pd.DataFrame(rows)

lote_files = build_lote_table(RAW_STJ, METADATA_RAW, ANOS_ANALISE)
if MAX_LOTES is not None:
    lote_files = lote_files.head(MAX_LOTES).copy()

PROCESS_SPINE_PATH = PROCESSED_DATA / 'stj_processos_ciclo_vida.parquet'

print('Lotes encontrados:', len(lote_files))
if not lote_files.empty:
    display(lote_files.assign(metadata_path=lote_files['metadata_path'].astype(str), zip_path=lote_files['zip_path'].astype(str)).head(20))
    print('Com metadata e ZIP:', int((lote_files['tem_metadata'] & lote_files['tem_zip']).sum()))
    print('Sem ZIP:', int((lote_files['tem_metadata'] & ~lote_files['tem_zip']).sum()))
    print('Sem metadata:', int((~lote_files['tem_metadata'] & lote_files['tem_zip']).sum()))
print('PROCESS_SPINE_PATH:', PROCESS_SPINE_PATH, PROCESS_SPINE_PATH.exists())

Lotes encontrados: 28


,lote,data_lote,ano_lote,metadata_path,zip_path,tem_metadata,tem_zip
0,202401,2024-01-10,2024,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras_metadata/2024/metadados20240110.json,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras/2024/textos20240110.zip,True,True
1,202402,2024-02-01,2024,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras_metadata/2024/metadados20240201.json,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras/2024/textos20240201.zip,True,True
2,202403,2024-03-01,2024,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras_metadata/2024/metadados20240301.json,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras/2024/textos20240301.zip,True,True
3,202404,2024-04-02,2024,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras_metadata/2024/metadados20240402.json,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras/2024/textos20240402.zip,True,True
4,202405,2024-05-02,2024,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras_metadata/2024/metadados20240502.json,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras/2024/textos20240502.zip,True,True
5,202406,2024-06-03,2024,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras_metadata/2024/metadados20240603.json,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras/2024/textos20240603.zip,True,True
6,202407,2024-07-01,2024,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras_metadata/2024/metadados20240701.json,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras/2024/textos20240701.zip,True,True
7,202408,2024-08-01,2024,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras_metadata/2024/metadados20240801.json,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras/2024/textos20240801.zip,True,True
8,202409,2024-09-02,2024,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras_metadata/2024/metadados20240902.json,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras/2024/textos20240902.zip,True,True
9,202410,2024-10-01,2024,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras_metadata/2024/metadados20241001.json,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras/2024/textos20241001.zip,True,True


Com metadata e ZIP: 28
Sem ZIP: 0
Sem metadata: 0
PROCESS_SPINE_PATH: /content/drive/MyDrive/Mestrado/2026/llms/data/processed/stj_processos_ciclo_vida.parquet False


In [5]:
lote_files = lote_files[lote_files['tem_metadata'] & lote_files['tem_zip']].copy()
if lote_files.empty:
    raise FileNotFoundError(
        f'Nenhum par metadados*.json + ZIP encontrado para ANOS_ANALISE={ANOS_ANALISE}. '
        f'Confira METADATA_RAW={METADATA_RAW} e RAW_STJ={RAW_STJ}.'
    )

lote_files = lote_files.sort_values('data_lote').reset_index(drop=True)
print('Lotes pareados para processamento:', len(lote_files))
print('Primeiro lote:', lote_files.loc[0, 'lote'])
print('Ultimo lote:', lote_files.loc[len(lote_files) - 1, 'lote'])

Lotes pareados para processamento: 28
Primeiro lote: 202401
Ultimo lote: 202604


## 3. Funcoes de leitura e normalizacao

In [6]:
def only_digits(value: Any) -> str | None:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    digits = re.sub(r'\D', '', str(value))
    return digits or None

def normalize_cnj(value: Any) -> str | None:
    digits = only_digits(value)
    return digits if digits and len(digits) == 20 else None

def parse_stj_datetime(value: Any) -> pd.Timestamp:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return pd.NaT
    text = str(value).strip()
    if not text:
        return pd.NaT
    if re.fullmatch(r'\d{14}', text):
        return pd.to_datetime(text, format='%Y%m%d%H%M%S', errors='coerce')
    if re.fullmatch(r'\d{8}', text):
        return pd.to_datetime(text, format='%Y%m%d', errors='coerce')
    return pd.to_datetime(text, errors='coerce', utc=False)

def normalize_seq_documento(value: Any) -> str | None:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    text = str(value).strip()
    text = re.sub(r'\.0$', '', text)
    return text or None

def normalize_process_text_key(value: Any) -> str | None:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    text = re.sub(r'\s+', ' ', str(value)).strip()
    if not text or text.lower() in {'nan', 'none', '<na>'}:
        return None
    return text

def txt_stem(name: str) -> str:
    return Path(name).stem.strip()

def list_zip_txt_files(zip_path: Path) -> list[str]:
    with zipfile.ZipFile(zip_path) as zf:
        return sorted(name for name in zf.namelist() if not name.endswith('/') and name.lower().endswith('.txt'))

def read_txt_from_zip(zf: zipfile.ZipFile, file_name: str) -> str:
    raw = zf.read(file_name)
    for encoding in ['utf-8', 'latin-1', 'cp1252']:
        try:
            return raw.decode(encoding)
        except UnicodeDecodeError:
            continue
    return raw.decode('utf-8', errors='replace')

def clean_legal_text(text: str) -> str:
    if text is None:
        return ''
    text = str(text)
    if '<' in text and '>' in text:
        text = BeautifulSoup(text, 'html.parser').get_text('\n')
    text = text.replace('\xa0', ' ')
    text = re.sub(r'\r\n?', '\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def pick_first_existing(columns: list[str], candidates: list[str]) -> str | None:
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None

## 4. Carregar metadados e manifesto de TXT

In [7]:
def load_metadata_json(path: Path) -> pd.DataFrame:
    data = json.loads(Path(path).read_text(encoding='utf-8'))
    if isinstance(data, list):
        return pd.DataFrame(data)
    if isinstance(data, dict):
        list_values = [v for v in data.values() if isinstance(v, list)]
        return pd.DataFrame(list_values[0]) if len(list_values) == 1 else pd.json_normalize(data)
    raise ValueError(f'Formato inesperado: {type(data)}')


def load_lote_metadata(lotes: pd.DataFrame) -> pd.DataFrame:
    frames = []
    for _, lote in tqdm(lotes.iterrows(), total=len(lotes), desc='metadados'):
        frame = load_metadata_json(lote['metadata_path'])
        frame['lote'] = lote['lote']
        frame['data_lote'] = lote['data_lote']
        frame['metadata_path'] = str(lote['metadata_path'])
        frame['metadata_file'] = Path(lote['metadata_path']).name
        frame['zip_path'] = str(lote['zip_path'])
        frame['zip_file'] = Path(lote['zip_path']).name
        frames.append(frame)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

metadata_df = load_lote_metadata(lote_files)
print('metadata_df:', metadata_df.shape)
metadata_df.head()

metadados:   0%|          | 0/28 [00:00<?, ?it/s]

metadata_df: (93482, 21)


,seqDocumento,dataPublicacao,tipoDocumento,numeroRegistro,processo,dataRecebimento,dataDistribuicao,ministro,recurso,teor,descricaoMonocratica,assuntos,lote,data_lote,metadata_path,metadata_file,zip_path,zip_file,SeqDocumento,dataDistribuição,NM_MINISTRO
0,222429397.0,1704852000000,ACÓRDÃO,202302207112,AREsp 2395784,1687748400000,1.688353e+12,HERMAN BENJAMIN,AGRAVO INTERNO,Negando,None,10076;9992;10076;10502;13237,202401,2024-01-10,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras_metadata/2024/metadados20240110.json,metadados20240110.json,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras/2024/textos20240110.zip,textos20240110.zip,NaN,NaN,NaN
1,223607504.0,1704852000000,ACÓRDÃO,202301653318,AREsp 2372018,1684292400000,1.685588e+12,HERMAN BENJAMIN,AGRAVO INTERNO,Negando,None,10011;10011,202401,2024-01-10,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras_metadata/2024/metadados20240110.json,metadados20240110.json,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras/2024/textos20240110.zip,textos20240110.zip,NaN,NaN,NaN
2,223607418.0,1704852000000,ACÓRDÃO,202301594482,AREsp 2363224,1683946800000,1.684811e+12,HERMAN BENJAMIN,AGRAVO INTERNO,Negando,None,7617;7617,202401,2024-01-10,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras_metadata/2024/metadados20240110.json,metadados20240110.json,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras/2024/textos20240110.zip,textos20240110.zip,NaN,NaN,NaN
3,223607440.0,1704852000000,ACÓRDÃO,202301520073,AREsp 2362200,1683601200000,1.685070e+12,HERMAN BENJAMIN,AGRAVO INTERNO,Negando,None,10023;10023,202401,2024-01-10,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras_metadata/2024/metadados20240110.json,metadados20240110.json,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras/2024/textos20240110.zip,textos20240110.zip,NaN,NaN,NaN
4,223607269.0,1704852000000,ACÓRDÃO,202301520073,AREsp 2362200,1683601200000,1.685070e+12,HERMAN BENJAMIN,AGRAVO INTERNO,Negando,None,10023;10023,202401,2024-01-10,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras_metadata/2024/metadados20240110.json,metadados20240110.json,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras/2024/textos20240110.zip,textos20240110.zip,NaN,NaN,NaN


In [8]:
def build_txt_manifest(lotes: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, lote in tqdm(lotes.iterrows(), total=len(lotes), desc='zips'):
        zip_path = Path(lote['zip_path'])
        with zipfile.ZipFile(zip_path) as zf:
            for info in zf.infolist():
                if info.filename.endswith('/') or not info.filename.lower().endswith('.txt'):
                    continue
                rows.append({
                    'lote': lote['lote'],
                    'data_lote': lote['data_lote'],
                    'zip_path': str(zip_path),
                    'zip_file': zip_path.name,
                    'txt_path': info.filename,
                    'seq_documento': normalize_seq_documento(txt_stem(info.filename)),
                    'txt_size_bytes': info.file_size,
                })
    return pd.DataFrame(rows)

txt_manifest = build_txt_manifest(lote_files)
print('TXT nos ZIPs:', len(txt_manifest))
txt_manifest.head()

zips:   0%|          | 0/28 [00:00<?, ?it/s]

TXT nos ZIPs: 86574


,lote,data_lote,zip_path,zip_file,txt_path,seq_documento,txt_size_bytes
0,202401,2024-01-10,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras/2024/textos20240110.zip,textos20240110.zip,20240110/186053111.txt,186053111,1
1,202401,2024-01-10,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras/2024/textos20240110.zip,textos20240110.zip,20240110/187272682.txt,187272682,1
2,202401,2024-01-10,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras/2024/textos20240110.zip,textos20240110.zip,20240110/188927763.txt,188927763,1
3,202401,2024-01-10,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras/2024/textos20240110.zip,textos20240110.zip,20240110/188934005.txt,188934005,1
4,202401,2024-01-10,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras/2024/textos20240110.zip,textos20240110.zip,20240110/190955167.txt,190955167,1


## 5. Padronizar metadados de documento

A tabela `documentos_manifest` ainda nao carrega texto; ela apenas vincula metadados e arquivo TXT.

In [9]:
def build_documentos_manifest(metadata_df: pd.DataFrame, txt_manifest: pd.DataFrame) -> pd.DataFrame:
    columns = metadata_df.columns.tolist()
    processo_col = pick_first_existing(columns, ['numeroProcesso', 'numeroUnico', 'processo', 'Processo'])
    registro_col = pick_first_existing(columns, ['numeroRegistro', 'registro', 'numRegistro'])
    seq_col = pick_first_existing(columns, ['SeqDocumento', 'seqDocumento', 'idDocumento'])
    data_col = pick_first_existing(columns, ['dataPublicacao', 'dataDocumento', 'dataDisponibilizacao'])
    tipo_col = pick_first_existing(columns, ['tipoDocumento', 'tipo', 'descricaoTipoDocumento'])
    ministro_col = pick_first_existing(columns, ['ministro', 'NM_MINISTRO', 'relator'])
    teor_col = pick_first_existing(columns, ['teor', 'descricaoMonocratica'])

    if seq_col is None:
        raise KeyError('Nenhuma coluna SeqDocumento/seqDocumento/idDocumento encontrada nos metadados')

    docs = pd.DataFrame(index=metadata_df.index)
    docs['seq_documento'] = metadata_df[seq_col].map(normalize_seq_documento)
    docs['numero_processo'] = metadata_df[processo_col].map(normalize_cnj) if processo_col else pd.NA
    docs['numero_processo_original'] = metadata_df[processo_col].map(normalize_process_text_key) if processo_col else pd.NA
    docs['numero_registro_stj'] = metadata_df[registro_col].map(normalize_process_text_key) if registro_col else pd.NA
    docs['processo_agregacao'] = docs['numero_processo'].copy()
    docs['tipo_chave_processo'] = np.where(docs['numero_processo'].notna(), 'cnj', pd.NA)
    sem_cnj_com_registro = docs['processo_agregacao'].isna() & docs['numero_registro_stj'].notna()
    docs.loc[sem_cnj_com_registro, 'processo_agregacao'] = docs.loc[sem_cnj_com_registro, 'numero_registro_stj']
    docs.loc[sem_cnj_com_registro, 'tipo_chave_processo'] = 'registro_stj'
    sem_chave_com_original = docs['processo_agregacao'].isna() & docs['numero_processo_original'].notna()
    docs.loc[sem_chave_com_original, 'processo_agregacao'] = docs.loc[sem_chave_com_original, 'numero_processo_original']
    docs.loc[sem_chave_com_original, 'tipo_chave_processo'] = 'numero_original'
    docs['data_documento'] = metadata_df[data_col].map(parse_stj_datetime) if data_col else pd.NaT
    docs['tipo_documento'] = metadata_df[tipo_col] if tipo_col else pd.NA
    docs['ministro_documento'] = metadata_df[ministro_col] if ministro_col else pd.NA
    docs['teor_documento'] = metadata_df[teor_col] if teor_col else pd.NA
    docs['assuntos_raw'] = metadata_df['assuntos'] if 'assuntos' in metadata_df.columns else pd.NA
    docs['metadata_row_idx'] = metadata_df.index
    docs['lote'] = metadata_df['lote'] if 'lote' in metadata_df.columns else pd.NA
    docs['data_lote'] = metadata_df['data_lote'] if 'data_lote' in metadata_df.columns else pd.NaT
    docs['metadata_path'] = metadata_df['metadata_path'] if 'metadata_path' in metadata_df.columns else pd.NA
    docs['metadata_file'] = metadata_df['metadata_file'] if 'metadata_file' in metadata_df.columns else pd.NA
    docs['zip_path_metadata'] = metadata_df['zip_path'] if 'zip_path' in metadata_df.columns else pd.NA

    docs = docs.merge(txt_manifest, on=['lote', 'seq_documento'], how='left', indicator='txt_match')
    docs['tem_txt'] = docs['txt_match'].eq('both')
    docs = docs.drop(columns=['txt_match'])
    return docs

documentos_manifest = build_documentos_manifest(metadata_df, txt_manifest)
print('documentos_manifest:', documentos_manifest.shape)
print('Com TXT:', documentos_manifest['tem_txt'].sum(), 'de', len(documentos_manifest))
documentos_manifest.head()

documentos_manifest: (93482, 23)
Com TXT: 84224 de 93482


,seq_documento,numero_processo,numero_processo_original,numero_registro_stj,processo_agregacao,tipo_chave_processo,data_documento,tipo_documento,ministro_documento,teor_documento,assuntos_raw,metadata_row_idx,lote,data_lote_x,metadata_path,metadata_file,zip_path_metadata,data_lote_y,zip_path,zip_file,txt_path,txt_size_bytes,tem_txt
0,None,None,AREsp 2395784,202302207112,202302207112,registro_stj,NaT,ACÓRDÃO,HERMAN BENJAMIN,Negando,10076;9992;10076;10502;13237,0,202401,2024-01-10,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras_metadata/2024/metadados20240110.json,metadados20240110.json,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras/2024/textos20240110.zip,NaT,NaN,NaN,NaN,NaN,False
1,None,None,AREsp 2372018,202301653318,202301653318,registro_stj,NaT,ACÓRDÃO,HERMAN BENJAMIN,Negando,10011;10011,1,202401,2024-01-10,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras_metadata/2024/metadados20240110.json,metadados20240110.json,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras/2024/textos20240110.zip,NaT,NaN,NaN,NaN,NaN,False
2,None,None,AREsp 2363224,202301594482,202301594482,registro_stj,NaT,ACÓRDÃO,HERMAN BENJAMIN,Negando,7617;7617,2,202401,2024-01-10,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras_metadata/2024/metadados20240110.json,metadados20240110.json,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras/2024/textos20240110.zip,NaT,NaN,NaN,NaN,NaN,False
3,None,None,AREsp 2362200,202301520073,202301520073,registro_stj,NaT,ACÓRDÃO,HERMAN BENJAMIN,Negando,10023;10023,3,202401,2024-01-10,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras_metadata/2024/metadados20240110.json,metadados20240110.json,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras/2024/textos20240110.zip,NaT,NaN,NaN,NaN,NaN,False
4,None,None,AREsp 2362200,202301520073,202301520073,registro_stj,NaT,ACÓRDÃO,HERMAN BENJAMIN,Negando,10023;10023,4,202401,2024-01-10,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras_metadata/2024/metadados20240110.json,metadados20240110.json,/content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras/2024/textos20240110.zip,NaT,NaN,NaN,NaN,NaN,False


In [10]:
validation = {
    'metadata_rows': len(documentos_manifest),
    'metadata_seq_unicos': documentos_manifest['seq_documento'].nunique(dropna=True),
    'txt_files': len(txt_manifest),
    'txt_seq_unicos': txt_manifest['seq_documento'].nunique(dropna=True),
    'metadata_com_txt': int(documentos_manifest['tem_txt'].sum()),
    'metadata_sem_txt': int((~documentos_manifest['tem_txt']).sum()),
    'lotes_processados': int(lote_files['lote'].nunique()),
    'txt_sem_metadata': int((~pd.MultiIndex.from_frame(txt_manifest[['lote', 'seq_documento']]).isin(pd.MultiIndex.from_frame(documentos_manifest[['lote', 'seq_documento']]))).sum()),
    'processos_com_numero_cnj': int(documentos_manifest['numero_processo'].notna().sum()),
    'processos_com_chave_agregacao': int(documentos_manifest['processo_agregacao'].nunique(dropna=True)),
}
validation_df = pd.Series(validation, name='valor').reset_index()
validation_df.columns = ['metrica', 'valor']
display(validation_df)

,metrica,valor
0,metadata_rows,93482
1,metadata_seq_unicos,91626
2,txt_files,86574
3,txt_seq_unicos,86574
4,metadata_com_txt,84224
5,metadata_sem_txt,9258
6,lotes_processados,28
7,txt_sem_metadata,2350
8,processos_com_numero_cnj,0
9,processos_com_chave_agregacao,89873


## 6. Carregar textos

Esta etapa pode ser pesada. Use `MAX_DOCUMENTOS` para testar antes de processar tudo.

In [ ]:
docs_to_read = documentos_manifest[documentos_manifest['tem_txt']].copy()
if MAX_DOCUMENTOS is not None:
    if RANDOM_SAMPLE:
        docs_to_read = docs_to_read.sample(n=min(MAX_DOCUMENTOS, len(docs_to_read)), random_state=RANDOM_STATE)
    else:
        docs_to_read = docs_to_read.sort_values(['data_documento', 'lote', 'seq_documento']).head(MAX_DOCUMENTOS)

docs_to_read = docs_to_read.reset_index(drop=True)
print('Documentos que serao lidos:', len(docs_to_read))

Documentos que serao lidos: 84224


: 

In [12]:
def attach_texts(docs: pd.DataFrame) -> pd.DataFrame:
    rows = []
    if docs.empty:
        return pd.DataFrame()

    group_cols = ['zip_path'] if 'zip_path' in docs.columns else []
    if not group_cols:
        raise KeyError('Coluna zip_path ausente; o manifesto de TXT precisa indicar o ZIP de origem.')

    for zip_path_value, group in tqdm(docs.groupby('zip_path', dropna=True), desc='zips com texto'):
        zip_path = Path(zip_path_value)
        with zipfile.ZipFile(zip_path) as zf:
            for _, row in group.iterrows():
                item = row.to_dict()
                txt_path = item.get('txt_path')
                try:
                    texto_original = read_txt_from_zip(zf, txt_path)
                except Exception as exc:
                    item['texto_original'] = ''
                    item['texto_limpo'] = ''
                    item['erro_leitura_txt'] = str(exc)
                else:
                    item['texto_original'] = texto_original
                    item['texto_limpo'] = clean_legal_text(texto_original)
                    item['erro_leitura_txt'] = None
                rows.append(item)

    result = pd.DataFrame(rows)
    result['n_chars_original'] = result['texto_original'].fillna('').astype(str).str.len()
    result['n_chars_limpo'] = result['texto_limpo'].fillna('').astype(str).str.len()
    result['n_words_limpo'] = result['texto_limpo'].fillna('').astype(str).str.split().str.len()
    result['texto_vazio'] = result['texto_limpo'].fillna('').astype(str).str.strip().eq('')
    return result

documentos_texto = attach_texts(docs_to_read)
print('documentos_texto:', documentos_texto.shape)
documentos_texto[['lote', 'seq_documento', 'processo_agregacao', 'tipo_chave_processo', 'numero_processo', 'numero_registro_stj', 'tipo_documento', 'data_documento', 'n_words_limpo', 'texto_vazio']].head()

zips com texto:   0%|          | 0/25 [00:00<?, ?it/s]

: 

: 

## 7. Anexar espinha dorsal por processo

Se o notebook 06 foi executado, anexamos datas e indicadores processuais sem duplicar documentos.

In [ ]:
if PROCESS_SPINE_PATH.exists():
    process_spine = pd.read_parquet(PROCESS_SPINE_PATH)
    process_cols = [
        'numero_processo',
        'ano_origem_cnj',
        'segmento_cnj',
        'data_primeira_distribuicao_stj',
        'data_ajuizamento_datajud',
        'primeira_aparicao_corpus',
        'ja_existia_antes_da_primeira_aparicao_corpus',
        'classe_nome_datajud',
        'classes_stj',
        'relatores_stj',
    ]
    process_cols = [col for col in process_cols if col in process_spine.columns]
    documentos_texto = documentos_texto.merge(process_spine[process_cols], on='numero_processo', how='left')
    print('Espinha dorsal anexada:', PROCESS_SPINE_PATH)
else:
    print('Espinha dorsal nao encontrada. Rode o notebook 06 para enriquecer esta tabela.')

documentos_texto.head()

Espinha dorsal nao encontrada. Rode o notebook 06 para enriquecer esta tabela.


,seq_documento,numero_processo,numero_processo_original,numero_registro_stj,processo_agregacao,tipo_chave_processo,data_documento,tipo_documento,ministro_documento,teor_documento,assuntos_raw,metadata_row_idx,txt_path,txt_size_bytes,tem_txt,texto_original,texto_limpo,erro_leitura_txt,n_chars_original,n_chars_limpo,n_words_limpo,texto_vazio
0,364173416,None,HC 1079583,202600875480,202600875480,registro_stj,2026-03-19,DECISÃO,HERMAN BENJAMIN,Negando,"00287.03415.05566., 00287.03415.03420.",14,364173416.txt,2885.0,True,DECISÃO<br>Cuida-se de Habeas Corpus impetrado em favor de MARCELO LIMA RIBEIRO DOS SANTOS em que se aponta como ato coator o acórdão proferido pelo TRIBUNAL DE JUSTIÇA DO ESTA...,DECISÃO\nCuida-se de Habeas Corpus impetrado em favor de MARCELO LIMA RIBEIRO DOS SANTOS em que se aponta como ato coator o acórdão proferido pelo TRIBUNAL DE JUSTIÇA DO ESTADO...,None,2821,2776,432,False
1,361877489,None,RHC 232875,202600649416,202600649416,registro_stj,2026-03-19,DECISÃO,OG FERNANDES,Negando,"00287.03415.03435., 00287.03523.03546., 01209.04263.10925.10926.",22,361877489.txt,12197.0,True,DECISÃO<br>Trata-se de recurso ordinário em habeas corpus com pedido liminar interposto por RODRIGO VIEIRA DOS SANTOS contra acórdão prolatado pelo TRIBUNAL DE JUSTIÇA DO ESTAD...,DECISÃO\nTrata-se de recurso ordinário em habeas corpus com pedido liminar interposto por RODRIGO VIEIRA DOS SANTOS contra acórdão prolatado pelo TRIBUNAL DE JUSTIÇA DO ESTADO ...,None,11803,11696,1795,False
2,364198331,None,HC 1079814,202600879774,202600879774,registro_stj,2026-03-19,DECISÃO,HERMAN BENJAMIN,Negando,00287.03603.03607.03608.,23,364198331.txt,10300.0,True,DECISÃO<br>Cuida-se de Habeas Corpus impetrado em favor de EDSON LAURENTINO DA SILVA em que se aponta como ato coator o acórdão proferido pelo TRIBUNAL DE JUSTIÇA DO ESTADO DE ...,DECISÃO\nCuida-se de Habeas Corpus impetrado em favor de EDSON LAURENTINO DA SILVA em que se aponta como ato coator o acórdão proferido pelo TRIBUNAL DE JUSTIÇA DO ESTADO DE SÃ...,None,10040,9914,1522,False
3,362682988,None,HC 1068851,202600212625,202600212625,registro_stj,2026-03-19,DECISÃO,OG FERNANDES,Não Conhecendo,"00287.03385.05560.12194., 00287.03405.03406., 00287.03400.03402.",50,362682988.txt,21165.0,True,DECISÃO<br>Trata-se de habeas corpus com pedido liminar impetrado em favor de ALFREDO DIB NETO contra acórdão prolatado pelo TRIBUNAL DE JUSTIÇA DO ESTADO DO RIO DE JANEIRO.<br...,DECISÃO\nTrata-se de habeas corpus com pedido liminar impetrado em favor de ALFREDO DIB NETO contra acórdão prolatado pelo TRIBUNAL DE JUSTIÇA DO ESTADO DO RIO DE JANEIRO.\nDep...,None,20576,20298,3149,False
4,364213285,None,HC 1080767,202600946369,202600946369,registro_stj,2026-03-19,DECISÃO,HERMAN BENJAMIN,Negando,00287.03369.03372.,53,364213285.txt,3523.0,True,"DECISÃO<br>Cuida-se de Habeas Corpus impetrado em favor de SERGIO MENDES ROCHA FILHO, apontando constrangimento ilegal em razão de ato praticado por Ministro do STJ, no julgame...","DECISÃO\nCuida-se de Habeas Corpus impetrado em favor de SERGIO MENDES ROCHA FILHO, apontando constrangimento ilegal em razão de ato praticado por Ministro do STJ, no julgament...",None,3454,3382,504,False


## 8. Tabela por processo

A tabela agregada concatena textos por processo em ordem temporal. Ela e util para analises longitudinais, mas a tabela documento-texto deve continuar sendo a fonte principal para auditoria.

In [ ]:
def join_texts_ordered(group: pd.DataFrame) -> str:
    group = group.sort_values(['data_documento', 'seq_documento'])
    chunks = []
    for _, row in group.iterrows():
        header = f"\n\n===== DOCUMENTO {row.get('seq_documento')} | {row.get('data_documento')} | {row.get('tipo_documento')} =====\n"
        chunks.append(header + str(row.get('texto_limpo') or ''))
    return ''.join(chunks).strip()

if GERAR_TEXTO_POR_PROCESSO and 'processo_agregacao' in documentos_texto.columns:
    valid = documentos_texto.dropna(subset=['processo_agregacao']).copy()
    if valid.empty:
        textos_por_processo = pd.DataFrame()
    else:
        textos_por_processo = (
            valid.groupby('processo_agregacao', dropna=True)
            .agg(
                tipo_chave_processo=('tipo_chave_processo', 'first'),
                numero_processo=('numero_processo', 'first'),
                numero_registro_stj=('numero_registro_stj', 'first'),
                numero_processo_original=('numero_processo_original', 'first'),
                n_documentos_texto=('seq_documento', 'nunique'),
                primeira_data_documento=('data_documento', 'min'),
                ultima_data_documento=('data_documento', 'max'),
                tipos_documento=('tipo_documento', lambda s: ' | '.join(sorted(set(s.dropna().astype(str))))),
                total_words_limpo=('n_words_limpo', 'sum'),
            )
            .reset_index()
        )
        textos = pd.DataFrame([
            {'processo_agregacao': processo_key, 'texto_processo': join_texts_ordered(group)}
            for processo_key, group in valid.groupby('processo_agregacao', dropna=True)
        ])
        textos_por_processo = textos_por_processo.merge(textos, on='processo_agregacao', how='left')
else:
    textos_por_processo = pd.DataFrame()

print('textos_por_processo:', textos_por_processo.shape)
textos_por_processo.head()

textos_por_processo: (497, 11)


,processo_agregacao,tipo_chave_processo,numero_processo,numero_registro_stj,numero_processo_original,n_documentos_texto,primeira_data_documento,ultima_data_documento,tipos_documento,total_words_limpo,texto_processo
0,202401817306,registro_stj,None,202401817306,HC 915044,1,2026-03-19,2026-03-19,DECISÃO,6251,===== DOCUMENTO 326498583 | 2026-03-19 00:00:00 | DECISÃO =====\nDECISÃO\nTrata-se de habeas corpus sem pedido de liminar impetrado em favor de VALTER PAIVA FARIAS contra acórd...
1,202403945500,registro_stj,None,202403945500,HC 954157,1,2026-03-19,2026-03-19,DECISÃO,3298,===== DOCUMENTO 363398139 | 2026-03-19 00:00:00 | DECISÃO =====\nDECISÃO\nTrata-se de habeas corpus com pedido de liminar impetrado em favor de MAYRA RODRIGUES FERREIRA em que ...
2,202404796569,registro_stj,None,202404796569,REsp 2189260,1,2026-03-19,2026-03-19,DECISÃO,2887,"===== DOCUMENTO 326163717 | 2026-03-19 00:00:00 | DECISÃO =====\nDECISÃO\nMARCELO JOSÉ GREGOLIN ANACLETO e ARIMAR TRANSPORTES E MADEIRAS LTDA apresentam recurso especial, funda..."
3,202404817767,registro_stj,None,202404817767,HC 969613,1,2026-03-19,2026-03-19,DECISÃO,1590,===== DOCUMENTO 363913228 | 2026-03-19 00:00:00 | DECISÃO =====\nDECISÃO\nTrata-se de habeas corpus com pedido de liminar impetrado em favor de FABIO CORREA FERREIRA em que se ...
4,202500287750,registro_stj,None,202500287750,HC 978221,1,2026-03-19,2026-03-19,DECISÃO,1742,===== DOCUMENTO 362782400 | 2026-03-19 00:00:00 | DECISÃO =====\nDECISÃO\nCuida-se de Agravo Regimental interposto por ORLANDO FRANCISCO DE JESUS contra decisão de minha lavra ...


## 9. Salvar artefatos

In [ ]:
outputs = {}
manifest_path = PROCESSED_DATA / 'stj_documentos_manifest.parquet'
documentos_manifest.to_parquet(manifest_path, index=False)
outputs['manifesto_documentos'] = manifest_path

text_path = PROCESSED_DATA / 'stj_documentos_por_processo.parquet'
documentos_texto.to_parquet(text_path, index=False)
outputs['documentos_texto'] = text_path

if not textos_por_processo.empty:
    proc_text_path = PROCESSED_DATA / 'stj_textos_por_processo.parquet'
    textos_por_processo.to_parquet(proc_text_path, index=False)
    outputs['textos_por_processo'] = proc_text_path

validation_path = REPORTS_DATA / 'stj_documentos_texto_validation.csv'
validation_df.to_csv(validation_path, index=False)
outputs['validacao'] = validation_path

print('Arquivos salvos:')
for name, path in outputs.items():
    print(f'- {name}: {path}')

Arquivos salvos:
- manifesto_documentos: /content/drive/MyDrive/Mestrado/2026/llms/data/processed/stj_documentos_manifest.parquet
- documentos_texto: /content/drive/MyDrive/Mestrado/2026/llms/data/processed/stj_documentos_por_processo.parquet
- textos_por_processo: /content/drive/MyDrive/Mestrado/2026/llms/data/processed/stj_textos_por_processo.parquet
- validacao: /content/drive/MyDrive/Mestrado/2026/llms/data/reports/summaries/stj_documentos_texto_validation.csv


In [ ]:
summary_lines = [
    '# Documentos e textos por processo STJ',
    '',
    f'- Lotes processados: {lote_files["lote"].nunique():,}',
    f'- Metadados de documentos: {len(documentos_manifest):,}',
    f'- Documentos com TXT: {int(documentos_manifest["tem_txt"].sum()):,}',
    f'- Documentos lidos nesta execucao: {len(documentos_texto):,}',
    f'- Textos vazios: {int(documentos_texto["texto_vazio"].sum()):,}',
    f'- Processos com texto por CNJ: {documentos_texto["numero_processo"].nunique(dropna=True):,}',
    f'- Chaves de processo com texto: {documentos_texto["processo_agregacao"].nunique(dropna=True):,}',
]
if not textos_por_processo.empty:
    summary_lines.append(f'- Linhas agregadas por processo: {len(textos_por_processo):,}')
summary_lines.extend([
    '',
    '## Observacoes',
    '',
    '- A tabela documento-texto e a fonte principal para auditoria.',
    '- A tabela por processo concatena textos e deve ser usada com cuidado em analises longitudinais.',
    '- Quando nao ha numero CNJ completo, a agregacao usa registro STJ ou numero original como chave auxiliar.',
])
report_path = REPORTS_DATA / 'stj_documentos_texto_summary.md'
report_path.write_text('\n'.join(summary_lines), encoding='utf-8')
report_path

## 10. Proximos passos

- Validar manualmente uma amostra de documentos por processo.
- Comparar `data_documento`, `data_primeira_distribuicao_stj` e `ano_origem_cnj`.
- Criar recortes por tipo de documento antes de usar LLMs.
- Preservar sempre a chave `seq_documento` para rastreabilidade.